# Test `CAPEnewsindex` against `sample_articles.xlsx`

This notebook is a hand-checkable test of the `CAPEnewsindex` library. `sample_articles.xlsx`
has 24 articles across 2 countries (Nigeria, Kenya), 2 newspapers each, and 2 months each
(3 articles per newspaper-month). Every article was written to hit a known, deliberate
combination of Economic / Policy / Uncertainty terms from the **real, production**
keyword dictionary bundled with `CAPEnewsindex`, and each row's
`expected_is_epu` label was verified against that dictionary (see
`scripts not included here, but every row's note_for_reviewer column names the exact terms it hits`).

Every newspaper-month has exactly 3 articles: 1 EPU article in the first month, 2 in the
second. That makes the whole pipeline -- classification, counting, history fitting, and
scoring -- computable by hand:

- `S_it` (EPU share) = 1/3 then 2/3 for every newspaper
- population std of `[1/3, 2/3]` = 1/6
- `N_it` = `S_it / sigma_i` = 2.0 then 4.0
- `mu` (history mean of `M_t`) = mean(2.0, 4.0) = **3.0**
- `EPU_t` = `100 * M_t / mu` = **66.67** in month 1, **133.33** in month 2, for *both* countries

This notebook asserts every one of those numbers against what the library actually
computes. If it runs with no `AssertionError`, the installed `CAPEnewsindex` matches this
worked example exactly.

## Setup

If you installed the package with `pip install CAPEnewsindex`, skip the `sys.path` line below
and just `import CAPEnewsindex as ni`. The production keyword dictionary
ships with the package (`ni.dictionary.load()` with no path), so the only file this notebook
reads from the repo is `examples/sample_articles.xlsx`.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "examples" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))  # remove this line if CAPEnewsindex is pip-installed

import pandas as pd
import CAPEnewsindex as ni

print("CAPEnewsindex", ni.__version__)

## Load the sample articles

In [ ]:
articles = pd.read_excel(REPO_ROOT / "examples" / "sample_articles.xlsx")
articles["year_month"] = articles["date"].astype(str).str.slice(0, 7)
articles.head()

## Step 1 -- classify: does `match.is_epu` agree with every hand-labeled `expected_is_epu`?

In [ ]:
kw = ni.dictionary.load()  # production dictionary bundled with CAPEnewsindex
assert ni.dictionary.validate(kw) == [], "production dictionary should have no overlap/empty issues"

def classify(row):
    entry = kw[row["country"]]
    return ni.match.is_epu(row["title"], row["text"], entry.economic, entry.policy, entry.uncertainty)

articles["actual_is_epu"] = articles.apply(classify, axis=1)
mismatches = articles[articles["actual_is_epu"] != articles["expected_is_epu"]]
assert mismatches.empty, mismatches[["country", "site", "date", "title", "expected_is_epu", "actual_is_epu"]]
print(f"All {len(articles)} articles classified exactly as expected.")
articles[["country", "site", "date", "title", "expected_is_epu", "note_for_reviewer"]]

## Step 2 -- deduplicate (a no-op here: every URL is unique) and count per newspaper-month

In [ ]:
deduped = ni.dedup.dedupe(articles)
assert len(deduped) == len(articles)

counts = ni.index.count(deduped, kw)
counts

Expect `epu_articles = [1, 2]` and `all_articles = 3` for every one of the 4
newspapers' 2 months.

In [ ]:
assert set(counts["all_articles"]) == {3}
assert sorted(counts["epu_articles"]) == [1, 1, 1, 1, 2, 2, 2, 2]

## Step 3 -- fit the frozen history over both months

In [ ]:
history = ni.index.fit_history(counts, start="2024-01", end="2024-02")

for country in ("Nigeria", "Kenya"):
    assert history.mu[country] == 3.0, history.mu
    for site, sigma in history.sigma[country].items():
        assert abs(sigma - 1 / 6) < 1e-9, (country, site, sigma)

print("mu:", history.mu)
print("sigma:", history.sigma)

## Step 4 -- score both months

In [ ]:
scored = ni.index.score(counts, history).set_index(["country", "year_month"])

for country in ("Nigeria", "Kenya"):
    assert abs(scored.loc[(country, "2024-01"), "EPU"] - 200 / 3) < 1e-6
    assert abs(scored.loc[(country, "2024-02"), "EPU"] - 400 / 3) < 1e-6

print("All EPU values match the hand-computed 66.67 / 133.33.")
scored.reset_index()

## Step 5 -- (optional) history persistence and a trivial two-country aggregate

This exercises `History.save`/`History.load` and `aggregate.aggregate` on the same
tiny example, with Nigeria and Kenya given equal, constant GDP weight so the regional
z-score is just the simple average of the two (identical) country z-score series --
another number you can check by hand.

In [ ]:
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    history_path = Path(tmp) / "history.json"
    history.save(history_path)
    reloaded = ni.index.History.load(history_path)
    assert reloaded.mu == history.mu and reloaded.sigma == history.sigma

gdp = pd.DataFrame({
    "country": ["Nigeria", "Nigeria", "Kenya", "Kenya"],
    "year_month": ["2024-01", "2024-02", "2024-01", "2024-02"],
    "gdp": [50.0, 50.0, 50.0, 50.0],
})

regions = ni.aggregate.aggregate(
    scored.reset_index(),
    gdp,
    groups={"TestRegion": ["Nigeria", "Kenya"]},
    history=reloaded,
    include_africa=False,
)
regions

## Done

Every assertion above passed, which means the installed `CAPEnewsindex` reproduces this
worked example exactly: classification, newspaper-month counting, history fitting
(`sigma`, `mu`), scoring (`EPU_t`), history persistence, and GDP-weighted aggregation.